# 📊 02: Evaluate Simulation

Capability-gap evaluation workbench. Runs the trajectory-evaluator over the trajectories from `01_simulate.ipynb` and renders the **Capability Dashboard**: a single artifact answering: where is the model struggling, how strong is the evidence, and what should we inspect or probe next?

Two run knobs (`SIM_RUN_ID` and `REPORT_RUN_ID`) default to "latest" but can be pinned to historical run ids: see §⚙️ Configuration.

## 🔧 Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import getpass
import json
import logging
import os
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from usersim.cli._models import (
    REQUIRED_EVALUATOR_ALIASES,
    apply_model_overrides,
    bundled_models_path,
    default_models_path,
    describe_models_source,
    load_models_config,
    print_resolved_models,
    prompt_for_provider_keys,
    require_aliases,
    smoke_test_models_or_raise,
)
from usersim.cli._env import describe_loaded_env, load_local_env
from usersim.cli._pipeline import evaluate_dataframe, sample_trajectories
from usersim.engine.core.storage import (
    list_runs,
    read_partitioned_dataset,
    resolve_run_or_raise,
    run_subroot,
    write_run_locales_partition,
    write_run_partition,
)
from usersim.reporting import (
    EVAL_SAMPLE_MANIFEST_FILENAME,
    build_capability_report,
    print_available_runs,
    read_eval_sample_manifest,
    record_eval_sample_pass,
    write_capability_dashboard_artifacts,
)

logging.getLogger("usersim.engine").setLevel(logging.INFO)

# Credentials from .env.local / .env. Anything already exported wins, and any
# key still missing is prompted for by the Credentials cell below.
print("env:", describe_loaded_env(load_local_env()))

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = REPO_ROOT / "output"
TRAJECTORY_PATH = OUTPUT_DIR / "trajectories"
EVAL_PATH = OUTPUT_DIR / "evaluations"
REPORT_DIR_ROOT = OUTPUT_DIR / "report"

## 🗂️ Available Runs

A quick overview of trajectory runs already on disk and whether each has an evaluation / rendered report. Use the `run_id` column to pin `SIM_RUN_ID` / `REPORT_RUN_ID` in the next cell.

In [ ]:
print_available_runs(TRAJECTORY_PATH, EVAL_PATH, REPORT_DIR_ROOT)

## ⚙️ Configuration

In [ ]:
# --- Run selection (both default to "latest") -------------------------
SIM_RUN_ID:    str | None = None    # which simulation to evaluate; None = latest
REPORT_RUN_ID: str | None = None    # which run to render; None = latest

# --- Models ----------------------------------------------------------
# Switch providers by changing one word. The evaluator does not have to run
# on the same provider as the simulation being scored, and using a different
# one is a reasonable way to avoid a model grading its own family.
#
#   "auto"       $USERSIM_MODELS_CONFIG, then models.local.toml, then bundled
#   "nvidia"     build.nvidia.com            NVIDIA_API_KEY
#   "openai"     api.openai.com              OPENAI_API_KEY
#   "openrouter" openrouter.ai               OPENROUTER_API_KEY
MODELS_PROFILE = "auto"

_SHIPPED = bundled_models_path().parent
MODELS_CONFIG_PATH = {
    "auto": default_models_path(),
    "nvidia": bundled_models_path(),
    "openai": _SHIPPED / "models_openai.toml",
    "openrouter": _SHIPPED / "models_openrouter.toml",
}[MODELS_PROFILE]
print("models config:", describe_models_source(MODELS_CONFIG_PATH))

MODEL_OVERRIDES: dict[str, str] = {}   # e.g. {"evaluator_model": "gpt-5.6-terra"}

# --- Evaluator -------------------------------------------------------
# Axes registry (pin a subset to focus a partial re-eval, e.g. ["safety"]):
#   Universal (every probe):  helpfulness | accuracy | coherence | response_conciseness
#                             personalization | role_adherence | safety
#                             cultural_sensitivity | language_appropriateness
#   tool_calling only:        internal_architecture_leaking | tool_use_correctness
#   general_educational only: pedagogical_quality
AXES: list[str] | None = None   # None = all applicable for each row's probe_family
# Deterministic scorers to dispatch -- DERIVED from the capability registry, never
# hand-listed. The evaluator runs exactly the scorers it is handed and keys off
# nothing else, so a capability whose scorer is absent finds no evidence and renders
# as "Untested" (n=0) -- indistinguishable from a model that was never tested. That
# is how all 11 financial_services capabilities came back empty across a 13-locale
# run while the scorer sat registered and correctly wired to its axes. Pin an
# explicit subset only to focus a partial re-eval; build_evaluator_config_builder
# warns about whatever you leave out.
from usersim.taxonomy.capabilities import scorers_required_by_capabilities

SCORERS = list(scorers_required_by_capabilities())
EVAL_COLUMN = "assistant_eval"

# --- Sampling --------------------------------------------------------
EVAL_SAMPLE_MODE = "full"           # full | per_locale | per_locale_probe | matched_pair
EVAL_SAMPLE_N: int | None = None    # None = no cap; int = per-group cap for non-"full" modes
EVAL_SAMPLE_RANDOM_SEED = 42

# --- Locale subset (optional) ----------------------------------------
EVAL_LOCALES: list[str] | None = None #["en_US"] #None   # None = every locale; ["en_US"] (or any list) -> filter traj_df to that locale subset,

# --- Run-Evaluator toggles -------------------------------------------
OVERWRITE_EVAL_RUN   = True  # True forces re-eval; False loads existing if present
SKIP_MODEL_SMOKE_TEST = False  # True skips the pre-eval (model, provider) HTTP check

# --- Render-only mode (derived) --------------------------------------
# Pin REPORT_RUN_ID, leave SIM_RUN_ID=None → render-only (Credentials /
# Sampling / Evaluator cells short-circuit). Both pinned = evaluate one,
# render another.
_render_only = REPORT_RUN_ID is not None and SIM_RUN_ID is None
if _render_only:
    print(
        f"🔍 Render-only: REPORT_RUN_ID={REPORT_RUN_ID!r}, SIM_RUN_ID=None. "
        "Credentials / Sampling / Evaluator cells will skip."
    )

## 🔑 Credentials

API keys for the model providers. The first run prompts via `getpass` (no echo); subsequent runs see the keys in `os.environ` and pass through. Keys needed depend on which providers your `MODEL_OVERRIDES` route to: `prompt_for_provider_keys` walks the loaded config and asks only for what's missing.

In [ ]:
if _render_only:
    print("🔍 Render-only mode -- skipping credentials (no LLM calls upcoming).")
else:
    # 1) Build-time NVIDIA API key (build.nvidia.com).
    if not os.environ.get("NVIDIA_API_KEY"):
        key = getpass.getpass("Enter your NVIDIA API key (build.nvidia.com): ")
        if key:
            os.environ["NVIDIA_API_KEY"] = key
    print("NVIDIA_API_KEY configured:", bool(os.environ.get("NVIDIA_API_KEY")))

    # 2) Load models config + apply per-alias overrides so we know which
    #    providers we'll need credentials for.
    MODELS = load_models_config(MODELS_CONFIG_PATH)
    require_aliases(MODELS, required=REQUIRED_EVALUATOR_ALIASES, context="02_evaluate_simulation")
    if MODEL_OVERRIDES:
        MODELS = apply_model_overrides(MODELS, MODEL_OVERRIDES)

    # 3) Prompt for any provider env var that's needed by a CURRENTLY-USED
    #    alias and not already set (e.g. NVIDIA_INFERENCE_HUB_KEY when hub
    #    aliases are loaded). This is what was previously silently blocking
    #    the cell -- now it's in its own clearly-labeled cell.
    prompt_for_provider_keys(MODELS)

    # 4) Resolved state (post-override).
    print_resolved_models(MODELS)

## 👀 Sampling Preview

In [ ]:
if _render_only:
    print("🔍 Render-only mode -- skipping sampling.")
else:
    SIM_RUN_ID = resolve_run_or_raise(TRAJECTORY_PATH, SIM_RUN_ID, label="SIM_RUN_ID")
    # Decide whether the evaluator cell will run. Three reasons to fire:
    # (1) no eval data on disk yet, (2) explicit OVERWRITE_EVAL_RUN,
    # (3) EVAL_LOCALES is set (locale-targeted re-eval / additive write).
    _will_evaluate = (
        not run_subroot(EVAL_PATH, SIM_RUN_ID).exists()
        or OVERWRITE_EVAL_RUN
        or bool(EVAL_LOCALES)
    )
    print(
        f"SIM_RUN_ID = {SIM_RUN_ID} "
        f"({'will evaluate' if _will_evaluate else 'eval exists, will load existing'})"
    )

    if not _will_evaluate:
        # Fast-path: existing eval, no overwrite, no locale filter -> skip
        # the trajectory load + sampling entirely. The "Run Evaluator"
        # cell will load the existing eval rows.
        print(
            f"Eval data already exists for run={SIM_RUN_ID}; skipping sampling. "
            f"Set OVERWRITE_EVAL_RUN=True (or EVAL_LOCALES=[...]) to re-evaluate."
        )
    else:
        traj_df = read_partitioned_dataset(TRAJECTORY_PATH, run=SIM_RUN_ID)
        n_total_before_filter = len(traj_df)
        print(f"Loaded {n_total_before_filter} trajectories for run={SIM_RUN_ID}")

        # Locale-targeted re-eval: validate the requested locales are
        # actually on the trajectory parquet, then filter. Loud failure
        # on typos so we don't silently evaluate zero rows.
        if EVAL_LOCALES:
            available = set(traj_df["locale"].astype(str).unique())
            requested = set(EVAL_LOCALES)
            unknown = sorted(requested - available)
            if unknown:
                raise ValueError(
                    f"EVAL_LOCALES = {EVAL_LOCALES!r} contains locale(s) "
                    f"{unknown} not present in trajectory run={SIM_RUN_ID}. "
                    f"Available: {sorted(available)}"
                )
            traj_df = traj_df[traj_df["locale"].astype(str).isin(requested)]
            print(
                f"🎯 EVAL_LOCALES filter: {sorted(requested)} -> "
                f"{len(traj_df)}/{n_total_before_filter} trajectories"
            )
            if traj_df.empty:
                raise ValueError(
                    f"EVAL_LOCALES filter on run={SIM_RUN_ID} returned zero "
                    f"trajectories. Check the simulator output for {EVAL_LOCALES!r}."
                )

        eval_source_df = sample_trajectories(
            traj_df,
            mode=EVAL_SAMPLE_MODE, n=EVAL_SAMPLE_N, seed=EVAL_SAMPLE_RANDOM_SEED,
        )
        print(f"Selected {len(eval_source_df)} / {len(traj_df)} trajectories")
        display(eval_source_df.groupby(["locale", "probe_family"]).size().reset_index(name="n"))

        eval_report_dir = run_subroot(REPORT_DIR_ROOT, SIM_RUN_ID)
        eval_report_dir.mkdir(parents=True, exist_ok=True)
        # Audit-trail manifest: APPEND a pass record so a locale-targeted
        # re-eval preserves the original pass's metadata instead of
        # silently overwriting it. fresh=True for whole-run rewrites
        # (eval parquet is being rmtree'd; old passes describe data
        # that no longer exists). fresh=False (default) for locale-
        # targeted writes (other locales' eval data + their pass
        # records stay on disk).
        record_eval_sample_pass(
            eval_report_dir / EVAL_SAMPLE_MANIFEST_FILENAME,
            run_id=str(SIM_RUN_ID),
            mode=EVAL_SAMPLE_MODE,
            n=EVAL_SAMPLE_N,
            random_seed=EVAL_SAMPLE_RANDOM_SEED,
            eval_locales=EVAL_LOCALES,
            n_selected=len(eval_source_df),
            n_total=n_total_before_filter,
            trajectory_ids=eval_source_df["trajectory_id"].astype(str).tolist(),
            fresh=not bool(EVAL_LOCALES),
        )

## ▶️ Run Evaluator

In [ ]:
if _render_only:
    print("🔍 Render-only mode -- skipping evaluator.")
else:
    eval_run_root = run_subroot(EVAL_PATH, SIM_RUN_ID)
    # Same fast-path the sampling cell used: load existing only when
    # there's no overwrite flag AND no locale filter requesting a
    # targeted re-eval / additive write.
    _load_existing = (
        eval_run_root.exists()
        and not OVERWRITE_EVAL_RUN
        and not EVAL_LOCALES
    )
    if _load_existing:
        eval_df = read_partitioned_dataset(EVAL_PATH, run=SIM_RUN_ID)
        print(
            f"Loaded {len(eval_df)} existing eval rows from {eval_run_root} "
            f"(set OVERWRITE_EVAL_RUN=True or EVAL_LOCALES=[...] to re-evaluate)"
        )
    else:
        # Smoke-test the evaluator alias before burning LLM budget on the
        # scorer chain. Scoped to REQUIRED_EVALUATOR_ALIASES because the
        # other aliases (user / assistant / judge / summary / api_response)
        # are simulator-side and never invoked by the evaluator pipeline.
        if not SKIP_MODEL_SMOKE_TEST:
            smoke_test_models_or_raise(MODELS, aliases=REQUIRED_EVALUATOR_ALIASES)
        eval_df = evaluate_dataframe(
            eval_source_df,
            models=MODELS, scorers=SCORERS, axes=AXES,
            eval_column=EVAL_COLUMN,
        )
        if EVAL_LOCALES:
            # Locale-targeted write: rmtree only the EVAL_LOCALES
            # subdirs of the run (preserves any other locales' eval
            # data on disk for SIM_RUN_ID). Default overwrite=True
            # gives a clean rewrite of the targeted locales.
            write_run_locales_partition(
                eval_df, EVAL_PATH, SIM_RUN_ID, locales=EVAL_LOCALES,
            )
            print(
                f"Wrote {len(eval_df)} evaluator rows for locales "
                f"{sorted(EVAL_LOCALES)} under {EVAL_PATH}/run={SIM_RUN_ID}/ "
                f"(other locales' eval partitions preserved)."
            )
        else:
            # Whole-run write: rmtree the entire run subdir if
            # OVERWRITE_EVAL_RUN, then fresh write.
            write_run_partition(eval_df, EVAL_PATH, SIM_RUN_ID, overwrite=OVERWRITE_EVAL_RUN)
            print(f"Wrote {len(eval_df)} evaluator rows under {EVAL_PATH}/run={SIM_RUN_ID}/")

## 📈 Capability Dashboard

> **Render an existing run**: pin `SIM_RUN_ID` (and optionally `REPORT_RUN_ID`) to the target run id and Run All. The "Run Evaluator" cell auto-skips when its eval data already exists -- no LLM calls, no surprise re-eval. Set `OVERWRITE_EVAL_RUN=True` to force a fresh evaluator pass.

In [ ]:
REPORT_RUN_ID = resolve_run_or_raise(TRAJECTORY_PATH, REPORT_RUN_ID, label="REPORT_RUN_ID")

# Reuse the eval-side traj_df when the run ids match (common case);
# reload otherwise. Falls through to reload in render-only mode where
# traj_df was never created.
report_traj_df = (
    traj_df
    if (not _render_only and REPORT_RUN_ID == SIM_RUN_ID and "traj_df" in dir())
    else read_partitioned_dataset(TRAJECTORY_PATH, run=REPORT_RUN_ID)
)

eval_run_root = run_subroot(EVAL_PATH, REPORT_RUN_ID)
if not eval_run_root.exists():
    _evaluated_runs = [
        r for r in list_runs(TRAJECTORY_PATH)
        if run_subroot(EVAL_PATH, r).exists()
    ]
    raise FileNotFoundError(
        f"\n🛑 REPORT_RUN_ID={REPORT_RUN_ID!r} has no eval data on disk.\n"
        f"   Looked under: {eval_run_root}\n"
        f"   Runs with eval data: {_evaluated_runs or '(none yet)'}\n"
        f"   Fix: pin REPORT_RUN_ID to one of those, OR unset REPORT_RUN_ID\n"
        f"   and set SIM_RUN_ID={REPORT_RUN_ID!r} to evaluate this run first."
    )
report_eval_df = read_partitioned_dataset(EVAL_PATH, run=REPORT_RUN_ID)

report_dir = run_subroot(REPORT_DIR_ROOT, REPORT_RUN_ID)
report_dir.mkdir(parents=True, exist_ok=True)

# `sample_mode` is a property of the eval run, not the notebook
# session. Read it from the report run's manifest (written by the
# sampling cell), which preserves a pass-by-pass audit trail across
# locale-targeted re-evals -- so the dashboard's "Sample" pill
# reflects the cumulative eval state on disk rather than just the
# last sampling pass. ``consensus_mode`` returns the shared mode
# when all passes agree, ``"mixed"`` when they don't.
manifest_path = report_dir / EVAL_SAMPLE_MANIFEST_FILENAME
_eval_manifest = read_eval_sample_manifest(manifest_path)
sample_mode = _eval_manifest.consensus_mode if _eval_manifest else "unknown"

# `model_id` derives from the run manifest -- see build_capability_report docstring.
report = build_capability_report(
    report_traj_df, report_eval_df,
    eval_column=EVAL_COLUMN, run_id=REPORT_RUN_ID,
    sample_mode=sample_mode, trajectory_root=TRAJECTORY_PATH,
)
report.report_paths.update({"sample_manifest": str(manifest_path)})

index_path = write_capability_dashboard_artifacts(
    report, report_dir, repo_root=REPO_ROOT, build_react=True,
)
print(f"Wrote {index_path}")
report.print_actors()
display(HTML(index_path.read_text()))

## 🪪 Identity contradictions

A contradiction is an identity reply that says "No" to a question naming the model's own developer, then names that developer as its maker: "No, I was made by X's researchers" in reply to "Are you made by X?", where X made the model. The dashboard counts it against **Identity: confirms its developer**, and on a pressed follow-up it counts as giving in. The cell below lists each one with the question it answered; a run without identity rows lists none.

In [ ]:
from usersim.reporting import identity_contradictions

CONTRADICTIONS_SHOWN = 20  # rows of the listing to display

contradictions = identity_contradictions(report_traj_df, report_eval_df, eval_column=EVAL_COLUMN)
if contradictions.empty:
    print(f"No contradictions in run {REPORT_RUN_ID}.")
else:
    print(
        f"Contradictions in run {REPORT_RUN_ID}: {len(contradictions)} turns "
        f"in {contradictions['trajectory_id'].nunique()} conversations."
    )
    display(contradictions.groupby(["locale", "opening"]).size().rename("contradictions").reset_index())
    display(contradictions.head(CONTRADICTIONS_SHOWN))

## 🆚 Multi-Model Comparison

Side-by-side bake-off of every per-run report under `output/report/`. Auto-discovers all runs by default; pin / subset / reorder via `COMPARE_RUN_IDS`. Renders a self-contained interactive Vega-Lite dashboard: leaderboard, quality-vs-verbosity Pareto plot, locale-tabbed grouped bars, sim-health comparison: at `output/comparison/run=cmp_<epoch>/index.html`.

This section is read-only and cheap (no LLM calls), so it always runs: `_render_only` does not gate it.

In [ ]:
# Multi-model comparison config ------------------------------------------------
COMPARE_RUN_IDS: list[str] | None = None  # None = all runs in output/report/; or e.g. ["1778723989", "1778675383"]
COMPARE_LABELS: dict[str, str] = {}       # {run_id: short_label} override; else short model_id (e.g. {"1778675383": "GPT-5.4"})
COMPARE_LOCALES: list[str] | None = None  # None = every locale the selected runs report; or e.g.
                                          # ["en_US", "en_IN", "te_Telu_IN"] to subset AND order the axis.
                                          # NOTE: the leaderboard score is a mean over per-locale means,
                                          # so subsetting changes what each model is scored ON. That is the
                                          # point when you ask "who is best on the Indic locales", but two
                                          # comparisons over different locale sets are not comparable.
COMPARISON_OUT_DIR = OUTPUT_DIR / "comparison"  # written to output/comparison/run=cmp_<epoch>/


## 🔍 Discover Comparison Runs

In [ ]:
from usersim.reporting import print_comparison_runs, select_runs_for_comparison

# Discover all reports under REPORT_DIR_ROOT, then filter / reorder per COMPARE_RUN_IDS
# (raises FileNotFoundError if any pinned id has no manifest on disk).
_selected_paths = select_runs_for_comparison(REPORT_DIR_ROOT, compare_run_ids=COMPARE_RUN_IDS)
print_comparison_runs(_selected_paths)


## 🧮 Build Comparison Report

In [ ]:
import time

from usersim.reporting import (
    build_comparison_report,
    write_comparison_dashboard_artifacts,
)

# Hard-bail on the edge cases printed by the discovery cell above.
if len(_selected_paths) < 2:
    raise SystemExit(
        "Discovery cell found <2 reports: comparison requires at least 2. "
        "See the message printed above."
    )

_comparison_id = f"cmp_{int(time.time())}"
_report = build_comparison_report(
    _selected_paths,
    comparison_id=_comparison_id,
    labels=COMPARE_LABELS or None,
    locales=COMPARE_LOCALES or None,
)
print(f"Built comparison: {_comparison_id}: {len(_report.entries)} models, "
      f"{len(_report.locales)} locales, {len(_report.cells)} cells")

_comparison_dir = COMPARISON_OUT_DIR / f"run={_comparison_id}"
_index_path = write_comparison_dashboard_artifacts(_report, _comparison_dir)
print(f"\nWrote {_index_path}")
display(HTML(_index_path.read_text()))